# **Modelo clasificatorio de calidad de manzanas**

In [2]:
# Instalación de librerías básicas
%pip install torch torchvision pandas numpy matplotlib scikit-learn seaborn

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from  torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

In [32]:
def load_data_from_github():
    try:
        # Opción#1: URLs Raw de archivos de github
        dataset_url = "https://raw.githubusercontent.com/VivianaPM/DL-VisNIR-QualityClassifier/refs/heads/feature/model-apple/apple/dataset_manzanas.csv"
        rangos_url = "https://raw.githubusercontent.com/VivianaPM/DL-VisNIR-QualityClassifier/refs/heads/feature/model-apple/apple/AppleQualTags.csv"
        # Opcion#2: Archivos desde un link publico en la web de google
        # csv_path_apple = "url"
        # csv_ranks = "url"
        print("Cargando datos...")
        
        dataset = pd.read_csv(dataset_url, sep=',')
        rangos = pd.read_csv(rangos_url, sep=';')
        
        print(f"Dataset cargado: {dataset.shape}")
        print(f"Rangos cargados: {rangos.shape}")
        
        return dataset, rangos
        
    except Exception as e:
        print(f"Error: {e}")
        return None, None

In [33]:
#Opción#1: Cargar datos desde GitHub 
dataset_df, ranks_df = load_data_from_github()
# Opción#2:Cargar datos para archivos publicos en la web
# ranks_df = pd.read_csv(csv_ranks)
# dataset_df=  pd.read_csv(csv_path_apple)
print("Columnas en dataset_df:", dataset_df.columns.tolist())
print("Columnas en ranks_df:", ranks_df.columns.tolist())

# Si 'Dry matter' no existe, busca el nombre correcto
if 'Dry matter' not in dataset_df.columns:
    print("Busca la columna equivalente a 'Dry matter' en:")
    print(dataset_df.columns.tolist())

Cargando datos...
Dataset cargado: (240, 143)
Rangos cargados: (4, 3)
Columnas en dataset_df: ['Apple', 'Dry matter', '430', '434', '438', '442', '446', '450', '454', '458', '462', '466', '470', '474', '478', '482', '486', '490', '494', '498', '502', '506', '510', '514', '518', '522', '526', '530', '534', '538', '542', '546', '550', '554', '558', '562', '566', '570', '574', '578', '582', '586', '590', '594', '598', '602', '606', '610', '614', '618', '622', '626', '630', '634', '638', '642', '646', '650', '654', '658', '662', '666', '670', '674', '678', '682', '686', '690', '694', '698', '702', '706', '710', '714', '718', '722', '726', '730', '734', '738', '742', '746', '750', '754', '758', '762', '766', '770', '774', '778', '782', '786', '790', '794', '798', '802', '806', '810', '814', '818', '822', '826', '830', '834', '838', '842', '846', '850', '854', '858', '862', '866', '870', '874', '878', '882', '886', '890', '894', '898', '902', '906', '910', '914', '918', '922', '926', '930', 

In [34]:
# División a traves de categorias temporales de estratificación
def create_categories_stratification(dry_matter, num_bis):
    return pd.cut(dry_matter, bins=num_bis, labels=False)
# Aplicar categorias al datset
dataset_df['temp_stratify'] = create_categories_stratification(dataset_df['Dry matter'], num_bis=8)

In [35]:
# Recomendación: dividir el dataset antes de asignar caategorias
# Primera división: Train (70%) vs temporal(30%)
train_idx, temp_idx = train_test_split(
    dataset_df.index,
    test_size=0.3,
    random_state=42,
    stratify=dataset_df['temp_stratify']
)
# Segunda división: Test (15%) vs validación(15%)
test_idx, val_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    random_state=42,
    stratify=dataset_df.loc[temp_idx, 'temp_stratify']
)

In [36]:
# Eliminar columna temporal
dataset_df.drop('temp_stratify', axis=1, inplace=True)
# Asignación de categorias
def define_categories(dry_matter, rangos_df):
    for _, fila in rangos_df.iterrows():
        if fila['Min_DryM'] <= dry_matter <= fila['Max_DryM']:
            return fila['Category']
    return 'Desconocida'    

In [37]:
# Aplicar etiquetado por separado
# Train
X_train = dataset_df.loc[train_idx].drop('Dry matter',axis=1)
Y_train_dry = dataset_df.loc[train_idx, 'Dry matter']
Y_train = Y_train_dry.apply(lambda x: define_categories(x, ranks_df))
# Validation
X_val = dataset_df.loc[val_idx].drop('Dry matter',axis=1)
Y_val_dry = dataset_df.loc[val_idx, 'Dry matter']
Y_val = Y_val_dry.apply(lambda x: define_categories(x, ranks_df))
# Test
X_test = dataset_df.loc[test_idx].drop('Dry matter',axis=1)
Y_test_dry = dataset_df.loc[test_idx, 'Dry matter']
Y_test = Y_test_dry.apply(lambda x: define_categories(x, ranks_df))

In [38]:
# Verificar la distribución

print('*'*5 + "Distribucción categorias" + '*'*5)
print(f"Training: {Y_train.value_counts()}")
print(f"\nValidation: {Y_val.value_counts()}")
print(f"\nTest: {Y_test.value_counts()}")

print('\n' + '*'*5 + "Distribucción Dry matter" + '*'*5)
print("Training - Dry matter (numérico):")
print(f"  Count: {Y_train_dry.count()}")
print(f"  Mean: {Y_train_dry.mean():.2f}")
print(f"  Std: {Y_train_dry.std():.2f}")
print(f"  Min: {Y_train_dry.min():.2f}")
print(f"  Max: {Y_train_dry.max():.2f}")

print("\nValidation - Dry matter (numérico):")
print(f"  Count: {Y_val_dry.count()}")
print(f"  Mean: {Y_val_dry.mean():.2f}")
print(f"  Std: {Y_val_dry.std():.2f}")
print(f"  Min: {Y_val_dry.min():.2f}")
print(f"  Max: {Y_val_dry.max():.2f}")

print("\nTest - Dry matter (numérico):")
print(f"  Count: {Y_test_dry.count()}")
print(f"  Mean: {Y_test_dry.mean():.2f}")
print(f"  Std: {Y_test_dry.std():.2f}")
print(f"  Min: {Y_test_dry.min():.2f}")
print(f"  Max: {Y_test_dry.max():.2f}")

print('\n' + '*'*5 + "Consistencia de rangos" + '*'*5)
print(f"Dry matter global: {dataset_df['Dry matter'].min():.2f} - {dataset_df['Dry matter'].max():.2f}")
print(f"Training Range: {Y_train_dry.min():.2f} - {Y_train_dry.max():.2f}")
print(f"Validation Range: {Y_val_dry.min():.2f} - {Y_val_dry.max():.2f}")
print(f"Test Range: {Y_test_dry.min():.2f} - {Y_test_dry.max():.2f}")

# Verificar proporciones
print('\n' + '*'*5 + "Proporciones" + '*'*5)
print(f"Training: {len(train_idx)/len(dataset_df):.1%} ({len(train_idx)} muestras)")
print(f"Validation: {len(val_idx)/len(dataset_df):.1%} ({len(val_idx)} muestras)")
print(f"Test: {len(test_idx)/len(dataset_df):.1%} ({len(test_idx)} muestras)")


*****Distribucción categorias*****
Training: Dry matter
Estándar     109
Optima        54
Aceptable      5
Name: count, dtype: int64

Validation: Dry matter
Estándar     23
Optima       11
Aceptable     2
Name: count, dtype: int64

Test: Dry matter
Estándar     24
Optima       11
Aceptable     1
Name: count, dtype: int64

*****Distribucción Dry matter*****
Training - Dry matter (numérico):
  Count: 168
  Mean: 0.16
  Std: 0.01
  Min: 0.14
  Max: 0.17

Validation - Dry matter (numérico):
  Count: 36
  Mean: 0.15
  Std: 0.01
  Min: 0.14
  Max: 0.17

Test - Dry matter (numérico):
  Count: 36
  Mean: 0.15
  Std: 0.01
  Min: 0.13
  Max: 0.17

*****Consistencia de rangos*****
Dry matter global: 0.13 - 0.17
Training Range: 0.14 - 0.17
Validation Range: 0.14 - 0.17
Test Range: 0.13 - 0.17

*****Proporciones*****
Training: 70.0% (168 muestras)
Validation: 15.0% (36 muestras)
Test: 15.0% (36 muestras)


In [ ]:
# Aplicar etiquetado
# dataset_df['Category'] = dataset_df['Dry matter'].apply(
#     lambda x: define_categories(x, ranks_df)
# )

if 'Category' not in dataset_df.columns:
    # Aplicar etiquetado si no existe la columna Category
    dataset_df['Category'] = dataset_df['Dry matter'].apply(
        lambda x: define_categories(x, ranks_df)
    )